In [0]:
-- =============================================================================
-- SECTION 1: PRODUCER HEALTH
-- =============================================================================

-- PH-1  Bronze ingest rate — last 24 hours
-- Widget: Line chart  |  x=hour  y=events_ingested (primary), mb_ingested (secondary)
-- ─────────────────────────────────────────────────────────────────────────────
SELECT
  DATE_TRUNC('hour', ingest_timestamp)           AS hour,
  COUNT(*)                                        AS events_ingested,
  ROUND(SUM(LENGTH(raw_json)) / 1048576.0, 1)    AS mb_ingested
FROM wiki_poc.poc.bronze_recentchange_raw
WHERE ingest_timestamp >= NOW() - INTERVAL 24 HOURS
GROUP BY 1
ORDER BY 1;


-- PH-2  Staleness counter — minutes since last Bronze event
-- Widget: Counter  |  value=minutes_stale  |  alert visually if > 10
-- (Fargate stall alarm fires at 5 min via CloudWatch, but this gives
--  an at-a-glance read inside the dashboard)
-- ─────────────────────────────────────────────────────────────────────────────
SELECT
  MAX(ingest_timestamp)                                                  AS latest_event,
  TIMESTAMPDIFF(MINUTE, MAX(ingest_timestamp), CURRENT_TIMESTAMP())     AS minutes_stale
FROM wiki_poc.poc.bronze_recentchange_raw;


-- PH-3  Hourly ingest rate — last 7 days (trend / anomaly baseline)
-- Widget: Bar chart  |  x=day_hour  y=events_ingested
-- ─────────────────────────────────────────────────────────────────────────────
SELECT
  DATE_TRUNC('hour', ingest_timestamp)   AS day_hour,
  COUNT(*)                               AS events_ingested
FROM wiki_poc.poc.bronze_recentchange_raw
WHERE ingest_timestamp >= NOW() - INTERVAL 7 DAYS
GROUP BY 1
ORDER BY 1;
